# E13 — O vigia da direção

O capítulo anterior terminou numa pergunta que o esquecimento deixa de pé: apagar os dias apaga,
com eles, a ordem em que vieram. Este caderno mede se a **direção do tempo** pode ser vigiada com
garantia — e "com garantia" quer dizer, neste livro, o par declarado: quantos alarmes falsos num
mundo que não tem direção nenhuma, e quanto tempo para ver uma direção que existe.

O instrumento é o mesmo das outras travessias: uma estatística e um limiar. A estatística é o
terceiro momento do incremento, padronizado pela escala da janela **anterior** ao bloco — quem se
padroniza com os dias que está medindo já sabe a resposta antes de responder. A conta do nulo é
clara: se os incrementos fossem simétricos, a estatística teria desvio raiz de quinze sobre a
janela, e um limiar de três desvios prometeria dois vírgula sete alarmes por mil blocos.

O caderno mede três coisas: se o instrumento vê direção (invertendo a série, o sinal tem de
trocar), quanto o nulo de verdade gasta, e quanto tempo ele leva para ver a direção que o mundo
tem.


In [1]:
# <- brinque com: JANELA, Z, ASSIMETRIAS, SEMENTES, SERIES
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, direcao, graficos

JANELA = 252
Z = 3.0
ASSIMETRIAS = (0.0, 0.02, 0.05, 0.1, 0.2, 0.4)
SEMENTES = 40
DIAS_POR_MUNDO = 252 * 60
SEMENTE = 11
SERIES = ("sp500.csv", "ibov.csv", "btc.csv")
SIGMA = 0.01

CONTA = direcao.desvio_da_conta(JANELA)
LIMIAR = direcao.limiar(Z, JANELA)
assimetria_declarada = lambda a: (6.0 * a + 8.0 * a ** 3) / (1.0 + 2.0 * a ** 2) ** 1.5
print("janela %d | desvio do nulo pela conta %.4f | limiar de z=%.1f: %.4f" % (JANELA, CONTA, Z, LIMIAR))
print("o limiar promete %.5f alarmes por bloco, isto e, %.1f por mil blocos"
      % (2.0 * (1.0 - 0.9986501), 1000.0 * 2.0 * (1.0 - 0.9986501)))


janela 252 | desvio do nulo pela conta 0.2440 | limiar de z=3.0: 0.7319
o limiar promete 0.00270 alarmes por bloco, isto e, 2.7 por mil blocos


In [2]:
# O mundo simetrico: quanto o nulo gasta de verdade.
nulos = np.concatenate([direcao.blocos(direcao.assimetrico(DIAS_POR_MUNDO,
                                                          np.random.default_rng(SEMENTE + i), SIGMA, 0.0), JANELA)
                        for i in range(SEMENTES)])
resumo_nulo = direcao.resumo(nulos, LIMIAR)
print("blocos %d | media %+.4f | dispersao %.4f | a conta dizia %.4f" % (resumo_nulo["blocos"], resumo_nulo["media"],
      resumo_nulo["dispersao"], CONTA))
print("taxa de alarme medida %.5f por bloco | a conta prometia %.5f | razao %.1f"
      % (resumo_nulo["taxa"], 2.0 * (1.0 - 0.9986501), resumo_nulo["taxa"] / (2.0 * (1.0 - 0.9986501))))
LIMIAR_MEDIDO = Z * resumo_nulo["dispersao"]
print("o limiar honesto, na mesma promessa: %.4f (o da conta era %.4f)" % (LIMIAR_MEDIDO, LIMIAR))


blocos 2360 | media +0.0032 | dispersao 0.3238 | a conta dizia 0.2440
taxa de alarme medida 0.02712 por bloco | a conta prometia 0.00270 | razao 10.0
o limiar honesto, na mesma promessa: 0.9713 (o da conta era 0.7319)


In [3]:
# A direcao declarada: quanto tempo o vigia leva para ver.
linhas = []
for a in ASSIMETRIAS:
    vals = np.concatenate([direcao.blocos(direcao.assimetrico(DIAS_POR_MUNDO, np.random.default_rng(41 + i), SIGMA, a), JANELA)
                           for i in range(SEMENTES)])
    conta_ = direcao.resumo(vals, LIMIAR)
    medido = direcao.resumo(vals, LIMIAR_MEDIDO)
    linhas.append({"assimetria": assimetria_declarada(a), "estatistica": conta_["media"],
                   "dispersao": conta_["dispersao"], "acerto_com_a_conta": conta_["taxa"],
                   "acerto_com_o_medido": medido["taxa"]})
tabela = pd.DataFrame(linhas).set_index("assimetria")
print(tabela.round(4).to_string())
print()
print("no mundo simetrico o acerto e a taxa de alarme falso: %.3f com o limiar da conta, %.3f com o honesto"
      % (tabela.loc[0.0, "acerto_com_a_conta"], tabela.loc[0.0, "acerto_com_o_medido"]))


            estatistica  dispersao  acerto_com_a_conta  acerto_com_o_medido
assimetria                                                                 
0.000000         0.0041     0.3163              0.0246               0.0038
0.119920         0.1274     0.3208              0.0364               0.0076
0.298757         0.3117     0.3420              0.1064               0.0360
0.590206         0.6136     0.4104              0.3513               0.1801
1.126189         1.1788     0.6288              0.7483               0.5767
1.920129         2.0591     1.1846              0.9364               0.8644

no mundo simetrico o acerto e a taxa de alarme falso: 0.025 com o limiar da conta, 0.004 com o honesto


In [4]:
# A serie real: a direcao esta no dado, e ha quanto tempo.
linhas = []
for arquivo in SERIES:
    preco = np.log(dados.carregar_serie(arquivo).to_numpy())
    m = direcao.momento_amostral(preco)
    invertida = direcao.momento_amostral(direcao.invertida(preco))
    blocos_do_dado = direcao.blocos(direcao.passos(preco), JANELA)
    linhas.append({"serie": arquivo.replace(".csv", ""), "dias": m["dias"], "momento": m["momento"],
                   "razao": m["razao"], "invertida": invertida["momento"],
                   "blocos": blocos_do_dado.size,
                   "dispersao_dos_blocos": direcao.resumo(blocos_do_dado, LIMIAR_MEDIDO)["dispersao"],
                   "acerto_com_o_medido": direcao.resumo(blocos_do_dado, LIMIAR_MEDIDO)["taxa"],
                   "acerto_com_a_conta": direcao.resumo(blocos_do_dado, LIMIAR)["taxa"]})
reais = pd.DataFrame(linhas).set_index("serie")
print(reais.round(4).to_string())
print()
print("a direcao do dado real, em desvios: %s" % {s: round(v, 1) for s, v in reais["razao"].items()})


       dias  momento    razao  invertida  blocos  dispersao_dos_blocos  acerto_com_o_medido  acerto_com_a_conta
serie                                                                                                          
sp500  6718  -0.3485  -7.3758     0.3485      25                5.1706                 0.32                0.32
ibov   6620  -0.3474  -7.2980     0.3474      25                2.2883                 0.16                0.20
btc    4387  -0.7042 -12.0433     0.7042      16                1.4666                 0.25                0.25

a direcao do dado real, em desvios: {'sp500': -7.4, 'ibov': -7.3, 'btc': -12.0}


In [5]:
# Figura 1: a estatistica do indice, ano a ano, com o limiar honesto.
preco_indice = np.log(dados.carregar_serie(SERIES[0]).to_numpy())
blocos_indice = direcao.blocos(direcao.passos(preco_indice), JANELA)
anos = np.arange(JANELA, JANELA * (blocos_indice.size + 1), JANELA) / JANELA
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
eixo.bar(anos, blocos_indice, 0.8, color=np.where(np.abs(blocos_indice) > LIMIAR_MEDIDO, "#b03a2e", "#1f4e79"))
eixo.axhline(LIMIAR_MEDIDO, color="#555555", ls="--", lw=1.2, label="limiar honesto")
eixo.axhline(-LIMIAR_MEDIDO, color="#555555", ls="--", lw=1.2)
eixo.set_xlabel("ano da série (blocos de %d dias)" % JANELA)
eixo.set_ylabel("terceiro momento dos incrementos")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E13_vigia_da_direcao", 1)
plt.close(fig)
print("blocos do indice: %d | acima do limiar honesto: %d" % (blocos_indice.size, int((np.abs(blocos_indice) > LIMIAR_MEDIDO).sum())))


blocos do indice: 25 | acima do limiar honesto: 8


In [6]:
# Figura 2: o acerto contra a direcao declarada, nos dois limiares.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
eixo.plot(tabela.index, tabela["acerto_com_a_conta"], marker="o", color="#b03a2e", lw=1.6,
          label="limiar da conta (%.3f)" % LIMIAR)
eixo.plot(tabela.index, tabela["acerto_com_o_medido"], marker="s", color="#1f4e79", lw=1.6,
          label="limiar honesto (%.3f)" % LIMIAR_MEDIDO)
eixo.axhline(tabela.loc[0.0, "acerto_com_o_medido"], color="#555555", ls=":", lw=1.2,
             label="o gasto do nulo com o limiar honesto")
for valor, nome, cor in ((reais.loc["sp500", "momento"], "índice", "#2e7d32"),
                         (reais.loc["btc", "momento"], "bitcoin", "#7f8c8d")):
    eixo.axvline(abs(valor), color=cor, ls="-.", lw=1.2, label="a direção do %s" % nome)
eixo.set_xlabel("assimetria declarada do mundo")
eixo.set_ylabel("fração dos blocos em que o vigia vê direção")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E13_vigia_da_direcao", 2)
plt.close(fig)
print("acertos: %s" % [round(v, 3) for v in tabela["acerto_com_o_medido"]])


acertos: [0.004, 0.008, 0.036, 0.18, 0.577, 0.864]


## Leitura visual das figuras

Feita abrindo os .png com a ponte de visão (AGENTS.md §9). Observação, não número.

Figura 1 (a estatística do índice, ano a ano). Vinte e cinco barras, uma por bloco de duzentos e cinquenta e dois dias, com o eixo vertical indo do zero até vinte abaixo e as duas linhas tracejadas do limiar honesto. O corpo do gráfico mora inteiro abaixo da linha do zero: quase toda barra é negativa, e quase todas são curtas, de modo que a impressão que fica é a de um chão de barras pequenas com poucas descendo muito fundo. As barras vermelhas são as que cruzam o limiar, oito delas, e são justamente as profundas. O eixo engana com força aqui: a escala vertical foi esticada pelas barras extremas, então as duas linhas do limiar honesto ficam encostadas na linha do zero e a olho nu passam a impressão de coincidir com ela; para uma barra de tamanho médio, cruzar o limiar e sair de zero têm a mesma aparência. Também engana o rótulo do eixo horizontal: ele numera blocos, e o primeiro bloco já deixou um ano inteiro de história atrás de si, porque a estatística se padroniza pela janela anterior. A barra um não é o primeiro ano da série, é o primeiro ano que o vigia consegue julgar.

Figura 2 (o acerto contra a direção declarada). O eixo horizontal é a assimetria que o mundo declara, de zero a dois; o vertical, de zero a pouco menos de um, é a fração de blocos em que o vigia vê direção. As duas curvas ficam coladas no chão até perto de três décimos e só então começam a subir, e a vermelha, do limiar da conta, fica acima da azul, do limiar honesto, em todo o trecho, com o vão entre as duas maior no meio do gráfico e mais estreito nas pontas. A linha pontilhada do gasto do nulo é praticamente invisível no fundo do quadro, e é ela que diz o que aquele chão significa. As duas verticais apontam para onde o dado real cai: o índice no trecho em que a curva está apenas deixando o chão, e o bitcoin no meio da subida. Duas armadilhas de leitura. O rótulo do eixo vertical fala em blocos, mas o que está desenhado é uma fração entre zero e um, e um leitor apressado pode ler o topo como oitenta blocos. E a curva vermelha estar sempre por cima convida à conclusão de que o limiar da conta é o instrumento melhor, quando a altura a mais dela é exatamente o alarme falso que o mundo sem direção paga. A figura sozinha não diz isso; quem diz é a linha pontilhada no fundo.


In [7]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES = {0.0: "simétrico", 0.02: "dois_centesimos", 0.05: "cinco_centesimos", 0.1: "dez_centesimos",
         0.2: "vinte_centesimos", 0.4: "quarenta_centesimos"}
resultado = {
    "direcao_janela": int(JANELA),
    "direcao_z": float(Z),
    "direcao_conta": float(CONTA),
    "direcao_limiar": float(LIMIAR),
    "direcao_sementes": int(SEMENTES),
    "direcao_blocos_por_mundo": int(DIAS_POR_MUNDO // JANELA),
    "direcao_blocos_nulo": int(resumo_nulo["blocos"]),
    "direcao_nulo_media": float(resumo_nulo["media"]),
    "direcao_nulo_dispersao": float(resumo_nulo["dispersao"]),
    "direcao_nulo_taxa": float(resumo_nulo["taxa"]),
    "direcao_conta_taxa": float(2.0 * (1.0 - 0.9986501)),
    "direcao_razao_das_taxas": float(resumo_nulo["taxa"] / (2.0 * (1.0 - 0.9986501))),
    "direcao_limiar_medido": float(LIMIAR_MEDIDO),
    "direcao_limiar_medido_razao": float(LIMIAR_MEDIDO / LIMIAR),
}
for a in ASSIMETRIAS:
    nome = NOMES[a]
    resultado["direcao_assimetria_%s" % nome] = float(assimetria_declarada(a))
    resultado["direcao_estatistica_%s" % nome] = float(tabela.loc[assimetria_declarada(a), "estatistica"])
    resultado["direcao_dispersao_%s" % nome] = float(tabela.loc[assimetria_declarada(a), "dispersao"])
    resultado["direcao_acerto_%s" % nome] = float(tabela.loc[assimetria_declarada(a), "acerto_com_a_conta"])
    resultado["direcao_acerto_honesto_%s" % nome] = float(tabela.loc[assimetria_declarada(a), "acerto_com_o_medido"])
for serie, curto in zip(SERIES, ("índice", "ibovespa", "bitcoin")):
    nome = serie.replace(".csv", "")
    resultado["direcao_%s_momento" % curto] = float(reais.loc[nome, "momento"])
    resultado["direcao_%s_razao" % curto] = float(reais.loc[nome, "razao"])
    resultado["direcao_%s_dias" % curto] = int(reais.loc[nome, "dias"])
    resultado["direcao_%s_invertida" % curto] = float(reais.loc[nome, "invertida"])
    resultado["direcao_%s_acerto_honesto" % curto] = float(reais.loc[nome, "acerto_com_o_medido"])
    resultado["direcao_%s_blocos" % curto] = int(reais.loc[nome, "blocos"])
    resultado["direcao_%s_dispersao_blocos" % curto] = float(reais.loc[nome, "dispersao_dos_blocos"])
resultado["direcao_indice_blocos_acima"] = int((np.abs(blocos_indice) > LIMIAR_MEDIDO).sum())

caminho = Path("lab/resultados/E13_vigia_da_direcao.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E13_vigia_da_direcao.json gravado | 66 grandezas
